# NTNU Workshop — Kelvin lattice: FEM → X-ray → neural_xray

Self-paced notebook (run **after** the 45 min demo). Pipeline:

1. **`fem_lattice_simulator`** — indent a Kelvin lattice, export deformed geometry per timestep  
2. **`xray_projection_render`** (`scripts/render_projections.sh`) → `data/kelvin/renders/`  
3. **Stage dataset** — build `transforms_00.json`, `transforms_XX.json`, `transforms_00_to_XX.json`  
4. **`neural_xray`** — train canonical volumes + velocity field

**Runtime:** T4 GPU. Total time is typically **1–3+ hours** (FEM + render + training).

Repo: [NTNU_metamaterials_workshop](https://github.com/igrega348/NTNU_metamaterials_workshop)

## Common errors

**Install failures** — see [`docs/COLAB_DEPENDENCIES.md`](../docs/COLAB_DEPENDENCIES.md). Cell 1 runs `scripts/install_colab_deps.sh` with pinned versions in `requirements-colab.txt`. It must end with **`All imports OK`** and **`numpy: 1.26.4`**. Use **Python 3.12** + **T4** (runtime 2025.10 or 2026.01).

**`ERROR: pip's dependency resolver`** — Colab’s pre-installed packages (tensorflow, jax 0.7, …) conflict on paper; ignore if the script finishes with `All imports OK`.

**`KeyError: 'optimizers'`** — delete partial velocity-field checkpoints and re-run training:

```bash
rm -rf /content/NTNU_metamaterials_workshop/outputs/kelvin/xray_vfield/
```

In [ ]:
# @title 1. Install dependencies (pinned stack — see docs/COLAB_DEPENDENCIES.md)

import os
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Python {sys.version_info.major}.{sys.version_info.minor} — need 3.12. "
        "Runtime → Change runtime type → 2025.10 or 2026.01."
    )

%cd /content/

REPO = "/content/NTNU_metamaterials_workshop"
if os.path.isdir(REPO):
    !rm -rf {REPO}
!git clone --recurse-submodules -q https://github.com/igrega348/NTNU_metamaterials_workshop.git

# Use *this* kernel's Python (same as %pip); installs requirements-colab.txt then editables
os.environ["PYTHON"] = sys.executable
!bash {REPO}/scripts/install_colab_deps.sh {REPO}

# Go 1.22+ for render cell (install_colab_deps.sh installs to /usr/local/go)
os.environ["PATH"] = "/usr/local/go/bin:" + os.environ.get("PATH", "")

%load_ext tensorboard
!go version
print("Ready:", REPO)

In [ ]:
# @title 2. FEM — simulate Kelvin lattice indentation

import os
REPO = "/content/NTNU_metamaterials_workshop"
RUN_NAME = "workshop_colab"
FEM = f"{REPO}/fem_lattice_simulator"
RUN_DIR = f"{FEM}/runs/{RUN_NAME}"
os.makedirs(f"{RUN_DIR}/model", exist_ok=True)
os.makedirs(f"{RUN_DIR}/timesteps", exist_ok=True)
os.makedirs(f"{RUN_DIR}/yaml", exist_ok=True)

%cd {FEM}

# Smaller mesh / fewer steps than desktop run_pipeline.sh (Colab-friendly)
!python scripts/generate_lattice_from_yaml.py \
  --yaml lattice.yaml --out {RUN_DIR}/model/out.json \
  --nx 4 --ny 4 --nz 4 --subdivide 4

!python scripts/apply_indent_boundary_conditions.py \
  --in {RUN_DIR}/model/out.json --out {RUN_DIR}/model/out.json \
  --patch-cells-x 2 --patch-cells-y 2 --patch-placement center \
  --indent-uz -0.8 --indenter-uxuy-zero

# CPU only: Colab jax_cuda12_plugin (0.7.x) conflicts with pinned jaxlib 0.6.2
!env JAX_PLATFORMS=cpu python -m src.main {RUN_DIR}/model/out.json \
  --ramp-steps 10 --output-prefix {RUN_DIR}/timesteps/{RUN_NAME} --output-every 2

!python scripts/json_to_yaml.py "{RUN_DIR}/timesteps/{RUN_NAME}_t*.json" \
  --radius-from-area --outdir {RUN_DIR}/yaml --overwrite

print("FEM YAML timesteps in", f"{RUN_DIR}/yaml")

In [ ]:
# @title 3. Render X-ray projections (xray_projection_render)

import glob, os, shutil
REPO = "/content/NTNU_metamaterials_workshop"
RUN_NAME = "workshop_colab"
DATA = f"{REPO}/data/kelvin"
YAML_DST = f"{DATA}/yaml"
YAML_SRC = f"{REPO}/fem_lattice_simulator/runs/{RUN_NAME}/yaml"
os.makedirs(YAML_DST, exist_ok=True)

for f in glob.glob(f"{YAML_SRC}/*.yaml"):
    shutil.copy2(f, YAML_DST)
print("Copied", len(glob.glob(f"{YAML_DST}/*.yaml")), "YAML files to", YAML_DST)

os.environ["YAML_GLOB"] = f"{RUN_NAME}_t*.yaml"
os.environ["VOLUME_RES"] = "128"
os.environ["RESOLUTION"] = "256"
!bash {REPO}/scripts/render_projections.sh

print("Renders:", glob.glob(f"{DATA}/renders/*/transforms.json"))

In [ ]:
# @title 4. Combine projections + transforms for neural_xray

REPO = "/content/NTNU_metamaterials_workshop"
RUN_NAME = "workshop_colab"
DATA = f"{REPO}/data/kelvin"

!python {REPO}/scripts/stage_kelvin_for_nerf.py \
  --renders-dir {DATA}/renders \
  --yaml-dir {DATA}/yaml \
  --out-dir {DATA}

In [ ]:
# @title 5. Train neural_xray (~30+ min on T4)

REPO = "/content/NTNU_metamaterials_workshop"
!bash {REPO}/scripts/train_kelvin_workshop.sh

In [ ]:
# @title 6. TensorBoard
%tensorboard --logdir /content/NTNU_metamaterials_workshop/outputs/kelvin/
